# Look At Action Designator

Demonstrates `LookAtActionDescription` — directing the robot's head/camera toward a target pose.

**Two approaches:**
- **Direct PyCRAM** — explicit target `PoseStamped`
- **LLM Pipeline (bridge)** — CRAM string with `(type LookAt)` → `SimulationBridge.execute`

**Prerequisites:** `cd cognitive_robot_abstract_machine && uv sync --active`

## 1 · Imports

In [1]:
import logging
logging.basicConfig(level=logging.WARNING)

from semantic_digital_twin.world import World
from semantic_digital_twin.robots.pr2 import PR2

from pycram.datastructures.dataclasses import Context
from pycram.datastructures.enums import Arms
from pycram.datastructures.pose import PoseStamped
from pycram.language import SequentialPlan
from pycram.motion_executor import simulated_robot
from pycram.testing import setup_world
from pycram.robot_plans import LookAtActionDescription

from llmr.serializers import SimulationBridge

print('All imports OK')

/home/malineni/workingdir/cognitive_robot_abstract_machine/semantic_digital_twin/src/semantic_digital_twin/robots/pr2.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Found these qp solvers: ['qpSWIFT', 'qpalm']


/home/malineni/workingdir/cognitive_robot_abstract_machine/probabilistic_model/src/probabilistic_model/probabilistic_circuit/rx/probabilistic_circuit.py:538: SyntaxWarning: invalid escape sequence '\_'
  """


All imports OK


## 2 · World & Context Setup

In [2]:
world = setup_world()

try:
    robot = PR2.from_world(world)
except Exception as exc:
    print(f'Note: PR2 full setup skipped ({type(exc).__name__}) — recovering annotation')
    robot = next((a for a in world.semantic_annotations if isinstance(a, PR2)), None)
    if robot is None:
        raise RuntimeError('Could not obtain PR2 annotation') from exc
    with world.modify_world():
        robot._setup_velocity_limits()
        robot._setup_hardware_interfaces()
        robot._setup_joint_states()

context = Context(world, robot)
print('World ready')

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_link
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_link
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_bellow_link
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name pr2/base_link
INFO:semantic_digital_twin.world:Trying to add kinematic_stru

Note: PR2 full setup skipped (WorldEntityNotFoundError) — recovering annotation
World ready


## 3 · SimulationBridge Setup

In [3]:
bridge = SimulationBridge(world, robot)
print('SimulationBridge ready')

SimulationBridge ready


## 4 · Visualize in RViz2 (optional)

Skip this section if ROS2 is not available.

In [4]:
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
import threading
import rclpy

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [5]:
_tf_publisher  = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print('ROS2 publishers started')
print('  Fixed Frame : apartment/apartment_root')
print('  TF topic    : /tf')
print('  Marker topic: /semworld/viz_marker  (QoS Durability=Transient Local)')

ROS2 publishers started
  Fixed Frame : apartment/apartment_root
  TF topic    : /tf
  Marker topic: /semworld/viz_marker  (QoS Durability=Transient Local)


---
## 5 · Look At — Direct PyCRAM

Point the robot head toward the milk carton on the countertop.

In [6]:
# Use the live object pose from the world
milk_body   = world.get_body_by_name('milk.stl')
look_target = PoseStamped.from_spatial_type(milk_body.global_pose)

with simulated_robot:
    SequentialPlan(
        context,
        LookAtActionDescription(target=[look_target]),
    ).perform()

print('Look At (direct) done')

INFO:pycram.language:Executing SequentialNode
INFO:pycram.robot_plans.actions.base:Performing action LookAtAction
INFO:pycram.language:Executing SequentialNode


Look At (direct) done


---
## 6 · Look At — SimulationBridge (CRAM string)

The CRAM type `LookAt` is passed directly to the bridge — no LLM needed for simple head-pointing tasks.

In [6]:
cram_look = (
    '(an action (type LookAt) '
    '(object (:tag milk.stl (an object (type Artifact size small color white)))))'
)

with simulated_robot:
    bridge.execute(cram_look)

print('Look At (bridge) done')

INFO:pycram.language:Executing SequentialNode
INFO:pycram.robot_plans.actions.base:Performing action LookAtAction
INFO:pycram.language:Executing SequentialNode


Look At (bridge) done


---
## Shutdown ROS2 Node

In [ ]:
try:
    _ros_node.destroy_node()
    rclpy.shutdown()
    print('ROS2 node stopped')
except Exception:
    print('(ROS2 node was not running)')